

# FINAL GCP Guide F26 (Fall 2026)
### OPIM 5512 — Applied Data Science · Module3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module3/FINAL_GCP_Guide_F26.ipynb)

***Do NOT click Run all.** Copy each gray cell into **Cloud Shell** (in the Google Cloud Console), top to bottom.*

# 🌐 GCP + GitHub Actions Setup Guide
**Dr. Dave Wanik and Rohit Akole** - Dept. Operations and Information Management - University of Connecticut

**Fall 2026 edition (F26)**, updated 9/28/2026 after a full test run on a brand-new GCP account.

**How to use this guide:** copy each gray code cell and paste it into **Cloud Shell**, top to bottom. (Colab may underline the gray cells in red: ignore that. Colab thinks they are Python, but they are Cloud Shell commands.) Plan on 30 to 45 minutes. Expect a bug or two and debug them with Claude. That is part of the lab.

**How long things take:** Cloud Shell can be slow to start (up to a minute). Turning on the APIs takes 1 to 2 minutes per cell. Each GitHub workflow takes about 2 to 3 minutes. The first scraper run takes 1 to 2 minutes. Waiting is normal; if something takes more than 5 minutes, ask Claude.

Recreate the Cloud Functions Gen2 + Cloud Scheduler Environment

This guide rebuilds the exact permissions, service accounts, and GitHub Actions integration from Dr. Wanik’s working scrapers, extractor (regex and LLM), materializer and ML models.
Students can follow these steps to get their own version running cleanly from a fork or new GCP project.

## 🤖 Stuck? Ask Claude what to do!

**Want a guide at your side the whole way?** Paste this into Claude before you start:

```
Here is my class's Google Cloud setup guide: https://github.com/drdave-teaching/OPIM5512-notebooks/blob/main/Module3/FINAL_GCP_Guide_F26.ipynb
Walk me through it one step at a time. Tell me exactly what to paste into Cloud Shell or click
in GitHub, wait for me to paste back what happened, and help me fix any errors before we move on.
```

Errors are normal here, and figuring them out is part of the lab. When something goes red, copy this prompt into Claude, fill in the blanks, and paste in the **whole** error message (a screenshot works too):

```
I'm setting up a Craigslist car scraper on Google Cloud, deployed with GitHub Actions, for my
OPIM 5512 class. I'm on Step ___ of the setup guide (___ step name ___).

I ran this in Cloud Shell (or: this GitHub Actions workflow failed):
___ paste the command or the workflow name ___

This is the full error:
___ paste the error ___

My project ID is ___ and my GitHub repo is ___/___.
What does this error mean, and what exactly should I type or click next?
```

Two rules: do the steps **in order** (most errors come from a skipped step or a Cloud Shell restart that wiped your variables), and **never paste passwords, keys, or tokens** into any chat. This setup does not use keys, so you should never need to.

# 0. Before You Start: Account, Budget, Repo, Cloud Shell

### 0a. Create your GCP project
1. Go to **console.cloud.google.com** and sign in with a personal Gmail account.
2. New to Google Cloud? Start the **free trial** (\$300 of credits for 90 days). Google asks for a card to verify you, but you are not charged unless you choose to upgrade.
3. Create a **new project**, for example `myscrapers-yournetid-f26`. Look at the grey **Project ID** under the name box: that ID is what you paste into the variables below. If the name is taken, Google adds a number to the ID (for example `myscrapers-dww05002-f26-510015`).
   - **Name** = just a label. **Project ID** = what you paste into this guide. **Project number** = all digits; you never need to look it up, Step 1 finds it for you.
   - To check your ID any time: click the project picker at the top of the console and read the **ID** column. Ignore **My First Project** (Google makes it automatically).
4. Make sure the project is linked to your billing account (**Billing > Account management** should list it).

### 0b. Set a \$300 budget (about 1 minute, do not skip)
1. Search **Budgets** in the console and open **Budgets & alerts**. Click **Create budget**.
2. **Scope:** give it a name, leave your project selected, and leave **Services** on **All services** (if it asks you to pick one, choose all of them: picking only Cloud Run would miss the storage and Vertex costs). Click **Next**.
3. **Amount:** choose **Specified amount** and type `300` (the value of your free credit). Click **Next**.
4. **Actions:** keep the alerts at 50%, 90%, and 100% (emails at \$150, \$270, and \$300). Optional early warning: click **Add threshold** and add **10%** (an email at \$30).
5. Optional: to send the alerts to a teammate too, check **Link Monitoring email notification channels to this budget**, pick your project, click **Manage notification channels**, add their email under **Email > Add new**, then come back, refresh, and select it.
6. Click **Finish**.

A budget only **sends emails**. It does not shut anything off. This scraper runs once an hour and saves at most 10 new listings, so it should cost pennies. On the free trial you are **not charged** when the credit runs out (things just stop until you upgrade). If you ever get a budget email, something is wrong: stop and ask for help.

### 0c. Get your own copy of the code on GitHub
1. Open **github.com/drdave-teaching/myscrapers** and click **Fork**.
2. You may rename it in the fork dialog (for example `myscrapers-f26`). Whatever the final name is, `GITHUB_REPO` below must match it **exactly**, as `your-username/your-repo-name`. If that name is taken in your account, GitHub quietly adds **-v2** (or similar): always copy the name from your browser's address bar.
3. In your fork, open the **Actions** tab and click **"I understand my workflows, go ahead and enable them."** Forks start with Actions turned off.

**Make it a real fork** (under your repo name GitHub shows *"forked from drdave-teaching/myscrapers"*). Do not download or copy the code instead: only a fork can pull in Dr. Wanik's fixes later.

**Getting Dr. Wanik's updates later:** when the course code changes, open your fork's **Code** tab and click **Sync fork → Update branch**. That also redeploys your functions automatically. If GitHub offers **Discard commits** instead, stop and ask first: that button throws away your own edits to the pipeline code. (Keep your own work in separate files, like notebooks, and leave the pipeline code alone.)

### 0d. Open Cloud Shell
Click the **`>_`** icon at the top right of the Cloud Console. The first time a `gcloud` command runs, a pop-up asks you to **Authorize** Cloud Shell. Click **Authorize**.

**If Cloud Shell disconnects** (it times out when idle), your variables are wiped. Paste the Step 1 variables cell again before you run anything else.

Example: Dr. Wanik's Fall 2026 project is `myscrapers-dww05002-f26` and his fork is `drdww/myscrapers-f26`.

# 🚀 1. Variables — Edit These First

Before running anything, open your Cloud Shell (top right in GCP console → terminal icon), then paste and modify this block:

In [ ]:
# ==========================================================
# 🧩 EDITABLE VARIABLES (change the first four lines)
# ==========================================================
PROJECT_ID="YOUR-PROJECT-ID"                # e.g. "myscrapers-dww05002-f26"
REGION="us-central1"                        # keep this
BUCKET_NAME="YOUR-PROJECT-ID"               # easiest: same as your project ID
GITHUB_REPO="YOUR-GITHUB-USER/YOUR-REPO"    # e.g. "drdww/myscrapers-f26" (must match your fork exactly)

# Internal naming (leave these alone)
WIF_POOL="github-pool"
WIF_PROVIDER="gh-actions"
RUNTIME_SA_ID="cf-runtime"
DEPLOYER_SA_ID="cf-deployer"
SCHED_SA_ID="scheduler-invoker"
# ==========================================================

# --- Derived values ---
gcloud config set project "${PROJECT_ID}"
PROJECT_NUMBER="$(gcloud projects describe "${PROJECT_ID}" --format='value(projectNumber)')"
RUNTIME_SA="${RUNTIME_SA_ID}@${PROJECT_ID}.iam.gserviceaccount.com"
DEPLOYER_SA="${DEPLOYER_SA_ID}@${PROJECT_ID}.iam.gserviceaccount.com"
SCHED_SA="${SCHED_SA_ID}@${PROJECT_ID}.iam.gserviceaccount.com"
SCHEDULER_AGENT="service-${PROJECT_NUMBER}@gcp-sa-cloudscheduler.iam.gserviceaccount.com"
PRINCIPAL_SET="principalSet://iam.googleapis.com/projects/${PROJECT_NUMBER}/locations/global/workloadIdentityPools/${WIF_POOL}/attribute.repository/${GITHUB_REPO}"

echo "✅ Using project ${PROJECT_ID} (${PROJECT_NUMBER})"
echo "✅ GitHub repo: ${GITHUB_REPO}"

**Check the output.** You should see your project ID followed by a long number in parentheses. If the parentheses are **empty**, stop: either the project ID is wrong, or you have not clicked **Authorize** yet. Fix it and paste the cell again.

Still blank after that? **Ask Claude** (see the prompt at the top) and paste in exactly what Cloud Shell printed.

# ⚙️ 2. Enable Required APIs

(idempotent — safe to re-run)

In [ ]:
gcloud services enable \
  cloudfunctions.googleapis.com \
  run.googleapis.com \
  cloudscheduler.googleapis.com \
  artifactregistry.googleapis.com \
  iam.googleapis.com \
  iamcredentials.googleapis.com \
  cloudbuild.googleapis.com \
  eventarc.googleapis.com \
  storage.googleapis.com \
  serviceusage.googleapis.com


A few more APIs...

In [ ]:
gcloud services enable \
  cloudresourcemanager.googleapis.com \
  pubsub.googleapis.com \
  logging.googleapis.com \
  compute.googleapis.com

# 👥 3. Create Service Accounts

(each Cloud Function will run as the runtime SA, and GitHub deploys using the deployer SA)

In [ ]:
gcloud iam service-accounts create "${RUNTIME_SA_ID}"  --display-name="Cloud Functions Runtime" || true
gcloud iam service-accounts create "${DEPLOYER_SA_ID}" --display-name="GitHub Deployer" || true
gcloud iam service-accounts create "${SCHED_SA_ID}"    --display-name="Scheduler Invoker" || true


# 🔗 4. Workload Identity Federation (WIF) Setup

This connects GitHub Actions → GCP securely (no JSON keys!).

In [ ]:
# Create pool (once per project)
gcloud iam workload-identity-pools create "${WIF_POOL}" \
  --location="global" \
  --display-name="GitHub Actions Pool" || true

# Create provider (connects this repo to GCP)
gcloud iam workload-identity-pools providers create-oidc "${WIF_PROVIDER}" \
  --workload-identity-pool="${WIF_POOL}" \
  --location="global" \
  --display-name="GitHub OIDC Provider" \
  --issuer-uri="https://token.actions.githubusercontent.com" \
  --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository,attribute.ref=assertion.ref" \
  --attribute-condition="attribute.repository=='${GITHUB_REPO}' && attribute.ref=='refs/heads/main'" || true


Grant GitHub Actions access to impersonate your deployer service

In [ ]:
gcloud iam service-accounts add-iam-policy-binding "${DEPLOYER_SA}" \
  --role="roles/iam.workloadIdentityUser" \
  --member="${PRINCIPAL_SET}"


# 🔒 5. Project-Level Roles for the Deployer SA

In [ ]:
for ROLE in roles/cloudfunctions.developer roles/run.admin roles/cloudscheduler.admin roles/artifactregistry.writer roles/serviceusage.serviceUsageAdmin; do
  gcloud projects add-iam-policy-binding "${PROJECT_ID}" \
    --member="serviceAccount:${DEPLOYER_SA}" \
    --role="${ROLE}"
done


# 🧩 6. Service Account-to-Service Account Relationships

Allow deployer to “act as” the runtime SA (needed to deploy Functions):

In [ ]:
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" \
  --member="serviceAccount:${DEPLOYER_SA}" \
  --role="roles/iam.serviceAccountUser"


Allow the Cloud Scheduler agent to mint OIDC tokens for the runtime SA (UPDATED MEMBER TO SCHED_SA):

In [ ]:
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" \
  --member="serviceAccount:${SCHED_SA}" \
  --role="roles/iam.serviceAccountTokenCreator"


### Allow Deployer to act as default service account user

In [ ]:
gcloud iam service-accounts add-iam-policy-binding \
  projects/-/serviceAccounts/${PROJECT_NUMBER}-compute@developer.gserviceaccount.com \
  --member="serviceAccount:${DEPLOYER_SA}" \
  --role="roles/iam.serviceAccountUser"


gcloud iam service-accounts add-iam-policy-binding ${RUNTIME_SA} \
  --member="serviceAccount:${DEPLOYER_SA}" \
  --role="roles/iam.serviceAccountAdmin"

A few more permissions

In [ ]:
# 1) Allow the deployer SA to use/impersonate the runtime SA
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" --member="serviceAccount:${DEPLOYER_SA}" --role="roles/iam.serviceAccountUser" --project="${PROJECT_ID}"

# 2) Allow Cloud Scheduler's *managed* service agent to mint tokens for the runtime SA
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" --member="serviceAccount:${SCHEDULER_AGENT}" --role="roles/iam.serviceAccountTokenCreator" --project="${PROJECT_ID}"

If the second command says the Cloud Scheduler service account **does not exist yet**, that is OK: skip it and keep going. The deploy workflow grants the same permission automatically the first time it runs.

### Added the Create Bucket if not exist condition

In [ ]:
if ! gcloud storage buckets describe "gs://${BUCKET_NAME}" >/dev/null 2>&1; then
  gcloud storage buckets create "gs://${BUCKET_NAME}" --project="${PROJECT_ID}" --location="${REGION}"
fi

#🪣 7. Grant Runtime SA Access to Storage Bucket

In [ ]:
gcloud storage buckets add-iam-policy-binding "gs://${BUCKET_NAME}" \
  --member="serviceAccount:${RUNTIME_SA}" \
  --role="roles/storage.objectAdmin"


# 🧱 8. Verify Everything


In [ ]:
gcloud iam service-accounts list

Verify Workload Identity Federation:

In [ ]:
gcloud iam workload-identity-pools list --location=global
gcloud iam workload-identity-pools providers list --workload-identity-pool="${WIF_POOL}" --location=global

Check IAM relationships:

In [ ]:
gcloud projects get-iam-policy "${PROJECT_ID}" | grep "${DEPLOYER_SA}" -A3
gcloud iam service-accounts get-iam-policy "${RUNTIME_SA}"

# 🧩 9. Add 7 Variables to Your GitHub Repo

**9a. Print your values.** Paste this cell into Cloud Shell. It prints the 7 variable names with your values filled in, ready to copy.

In [ ]:
# Uses the variables from Step 1 (Cloud Shell restarted? Paste the Step 1 cell again first.)
WORKLOAD_IDENTITY_PROVIDER="projects/${PROJECT_NUMBER}/locations/global/workloadIdentityPools/${WIF_POOL}/providers/${WIF_PROVIDER}"

if [ -z "${PROJECT_NUMBER}" ]; then
  echo "⚠️  Your project number is blank. Paste the Step 1 cell again, then paste this cell again."
fi

echo "=================================================================="
echo "GitHub > Settings > Secrets and variables > Actions > Variables tab"
echo "Add these 7 (click 'New repository variable' for each one):"
echo "=================================================================="
printf "%-28s %s\n" "BASE_SITE"                  "(your personal link from the BASE_SITE sheet on HuskyCT)"
printf "%-28s %s\n" "BUCKET_NAME"                "${BUCKET_NAME}"
printf "%-28s %s\n" "DEPLOYER_SA"                "${DEPLOYER_SA}"
printf "%-28s %s\n" "PROJECT_ID"                 "${PROJECT_ID}"
printf "%-28s %s\n" "REGION"                     "${REGION}"
printf "%-28s %s\n" "RUNTIME_SA"                 "${RUNTIME_SA}"
printf "%-28s %s\n" "WORKLOAD_IDENTITY_PROVIDER" "${WORKLOAD_IDENTITY_PROVIDER}"
echo
echo "✅ Step 9a done. Next: add these 7 in GitHub (Step 9b)."

**9b. Add them in GitHub.** In your repo, click **Settings**, then **scroll down the left sidebar** to the **Security and quality** section and click **Secrets and variables → Actions**. Open the **Variables** tab (not Secrets) and click **New repository variable** once for each row:

| Name | Value |
| --- | --- |
| `BASE_SITE` | **Your personal link** from the BASE_SITE sheet on HuskyCT (find your name). Paste it exactly. |
| `BUCKET_NAME` | from the Cloud Shell printout |
| `DEPLOYER_SA` | from the Cloud Shell printout |
| `PROJECT_ID` | from the Cloud Shell printout |
| `REGION` | `us-central1` |
| `RUNTIME_SA` | from the Cloud Shell printout |
| `WORKLOAD_IDENTITY_PROVIDER` | from the Cloud Shell printout (the long `projects/…/providers/gh-actions` line) |

**Add exactly these 7 and nothing else.** The workflows have built-in defaults for every other setting.

- **Settings → Actions**, higher up in the sidebar, is a different page. Your variables are not there.
- `BASE_SITE` is kept out of the code on purpose, so the site address never sits in a public repo. Everyone gets a **different** link (a different area and seller type), so each of you builds your own dataset, and later we can combine them into one big model. If `BASE_SITE` is missing, **Deploy Scraper** stops with a red X that tells you so.
- Not sure which value goes where? Paste the Cloud Shell printout into Claude and **ask Claude** to match them up.

When you are done, your Variables tab should look like this (your values will be different):

![The 7 GitHub Variables for the scraper](https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module3/img/F26_github_variables.png)

# ✅ 10. Test Your Setup

You do not need to push any code. In your fork on GitHub:

1. Open the **Actions** tab.
2. Click a workflow on the left, then **Run workflow** (branch: `main`) > **Run workflow**.
3. Start with **Deploy Scraper**, then the extractor, materialize-master, and train-dt workflows. The LLM extractor is optional.
4. Wait for each run to show a **green check** (about 2 to 4 minutes each). **Five green checks** means everything deployed. Four is fine if you skip the LLM extractor.

A red X? Click into the run and open the failed step. Then **ask Claude** with the prompt at the top of this guide, including the workflow name, the name of the step that failed, and the full error.

Then confirm the deployment from Cloud Shell:

In [ ]:
gcloud functions list --regions="${REGION}"
gcloud scheduler jobs list --location="${REGION}"


You should see all your scheduled functions enabled:

In [ ]:
extractor-per-listing-hourly
materialize-master-hourly
train-dt-hourly
craigslist-scraper-hourly


# Summary Design

In [ ]:
GitHub (OIDC) ───▶ cf-deployer (Workload Identity User)
                          │
                          ├─ deploys CF Gen2 + Scheduler
                          │
                          ▼
                    cf-runtime (ServiceAccountUser)
                          │
                          ├─ executes function logic
                          ├─ writes to GCS bucket
                          └─ trusted by Scheduler agent (TokenCreator)


# 🚗 Watch Your Bucket Fill Up

Everything runs on a schedule in **Cloud Scheduler**, every hour, Eastern time:

| Minute | Job | What it does |
| --- | --- | --- |
| :00 | `craigslist-scraper-hourly` | saves up to **10 NEW listings** into `scrapes/` (never the same car twice) |
| :10 | `extractor-per-listing-hourly` | pulls price, make, model, year, mileage into `structured/` |
| :12 | `extractor-llm-poc-hourly` | optional: the LLM version of the extractor |
| :15 | `materialize-master-hourly` | builds one tidy table, `listings_master.csv` |
| :20 | `train-dt-hourly` | retrains the decision tree and saves predictions for cars it has not predicted yet |

You do not have to wait for the hour:

1. In the Cloud Console, search **Cloud Scheduler**.
2. Check the box next to `craigslist-scraper-hourly` and click **Force run**.

   ![Cloud Scheduler: check the scraper job, then Force run](https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module3/img/F26_cloud_scheduler_force_run.png)

   **Tick the checkbox first** (the Force run button stays grey until you do), and look for the pop-up **"1 enabled job was forced to run."** Then wait about a minute and click **Refresh** until the status shows **Success**. (Cloud Shell alternative: `gcloud scheduler jobs run craigslist-scraper-hourly --location="${REGION}"`.)

3. **Check your scrapes.** Open **Cloud Storage > Buckets >** your bucket > **`scrapes/`**. You will see a folder named with the date and time (UTC), holding up to **10 `.txt` files** (one per car ad) plus `index.csv` (which ad each file came from). Click a `.txt` file and **Download** it: it is the full text of a real ad. Open the same ad on Craigslist and compare. Looking for a particular car? In Cloud Shell: `gcloud storage cat "gs://${BUCKET_NAME}/scrapes/*/index.csv" | grep -i sonata`. After the extractor runs, the same car's structured record is in `structured/run_id=<that folder>/jsonl/<same file name>.jsonl`.
4. Then Force-run `extractor-per-listing-hourly`, then `materialize-master-hourly`, one at a time, waiting for each to show **Success**. Your tidy table appears at `structured/datasets/listings_master.csv`.

**Force-run the scraper only once**, then run the extractor. The extractor reads the *newest* batch of listings, so two scraper runs in a row means the first batch never gets extracted.

**When do predictions show up?** The model trains on earlier days and predicts today's new cars, so it needs **at least 2 days** of listings (and 40+ cars from before today). Expect your first predictions tomorrow. They land in `structured/preds/`: one small file per hour with only the newly predicted cars, plus `structured/preds/preds_master.csv`, which keeps **one row per car** (the file to chart later).

**Account safety:** GitHub only *deploys* your code. The schedule lives on Google Cloud, and nothing in this setup commits to your GitHub repo on a timer. Keep it that way: do not add workflows that push to your repo on a schedule.

# 🤖11. Vertex Permissions (PoC) - do this too!

In [ ]:
# Uses the variables from Step 1 (paste that cell again first if Cloud Shell restarted)
echo "Project: ${PROJECT_ID}"
echo "Service Account to update: ${RUNTIME_SA}"

# Grant the Vertex AI User role to the runtime Service Account
gcloud projects add-iam-policy-binding "${PROJECT_ID}" \
    --member="serviceAccount:${RUNTIME_SA}" \
    --role="roles/aiplatform.user"

# Grant the Storage Object Creator role on the bucket
gcloud storage buckets add-iam-policy-binding "gs://${BUCKET_NAME}" \
    --member="serviceAccount:${RUNTIME_SA}" \
    --role="roles/storage.objectCreator"

Run the next cell **only after** the LLM extractor workflow has a green check, because that workflow is what creates the `extractor-llm-poc` service.

In [ ]:
gcloud run services update extractor-llm-poc \
    --region=us-central1 \
    --timeout="3600s" # 60 minutes max